# Deliverables 5 and 6

## PART 1
We wish to analyze the following genes:
- CYP2C8
- CYP2C9
- CYP2C19

Using UCSC Genome Browser, we can observe that they are located in the following positions within the hg38 version of the human genome, respectively
- chr10:95036772-95069497
- chr10:94938658-94990091
- chr10:94762681-94855547

They are all located within chromosome 10, so we download these region of the hg38 reference genome.

In [26]:
%%bash
# Downloading relevant chromosome from the reference genome
curl https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz > chr10.fa.gz
gunzip chr10.fa.gz

total 266560
-rw-r--r--  1 ashleyosuna  staff          0 Oct 26 09:58 ai.md
-rw-r--r--  1 ashleyosuna  staff  136473378 Oct 26 10:51 chr10.fa
drwxr-xr-x  2 ashleyosuna  staff         64 Oct 26 09:59 data
-rw-r--r--@ 1 ashleyosuna  staff       1237 Oct 26 10:56 week5.ipynb


## PART 2

In [46]:
%%bash
# aligning using minimap, then passing it to samtools to get bam and bai files
minimap2 -a chr10.fa data/illumina.fq | samtools view -bS | samtools sort -o illumina.sorted.bam
samtools index illumina.sorted.bam

minimap2 -a chr10.fa data/pacbio.fq | samtools view -bS | samtools sort -o pacbio.sorted.bam
samtools index pacbio.sorted.bam

[M::mm_idx_gen::2.275*0.91] collected minimizers
[M::mm_idx_gen::2.882*1.35] sorted minimizers
[M::main::2.883*1.35] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::3.093*1.33] mid_occ = 178
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::3.241*1.31] distinct minimizers: 16061920 (79.91% are singletons); average occurrences: 1.563; average spacing: 5.329; total length: 133797422
[M::worker_pipeline::9.787*1.83] mapped 309505 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -a chr10.fa data/illumina.fq
[M::main] Real time: 9.888 sec; CPU: 18.031 sec; Peak RSS: 1.240 GB
[M::mm_idx_gen::2.073*0.97] collected minimizers
[M::mm_idx_gen::2.685*1.43] sorted minimizers
[M::main::2.685*1.43] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::2.892*1.40] mid_occ = 178
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::3.039*1.38] distinct minimizers: 16061920 (79.91% are singletons); 

## PART 3: calling variants

In [56]:
%%bash
# FINDING VARIANTS
bcftools mpileup -f chr10.fa -r 'chr10:95036772-95069497','chr10:94938658-94990091','chr10:94762681-94855547' illumina.sorted.bam | bcftools call -mv -Ov -o illumina.vcf
bcftools mpileup -f chr10.fa -r 'chr10:95036772-95069497','chr10:94938658-94990091','chr10:94762681-94855547' pacbio.sorted.bam | bcftools call -mv -Ov -o pacbio.vcf

Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250
Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid
[mpileup] 1 samples in 1 input files
[mpileup] maximum number of reads per input file set to -d 250


## PART 4: phasing the variants

In [64]:
%%bash
extractHAIRS --bam illumina.sorted.bam --VCF illumina.vcf --ref chr10.fa --out illumina.frag
hapcut2 --fragments illumina.frag --VCF illumina.vcf --output illumina.hapcut

extractHAIRS --bam pacbio.sorted.bam --VCF pacbio.vcf --ref chr10.fa --out pacbio.frag
hapcut2 --fragments pacbio.frag --VCF pacbio.vcf --output pacbio.hapcut

rm illumina.frag pacbio.frag illumina.hapcut pacbio.hapcut


Extracting haplotype informative reads from bamfiles illumina.sorted.bam minQV 13 minMQ 20 maxIS 10

00 

VCF file illumina.vcf has 275 variants 
adding chrom chr10 to index 
vcffile illumina.vcf chromosomes 1 hetvariants 154 variants 275 
detected 4 variants with two non-reference alleles, these variants will not be phased
reading fasta index file chr10.fa.fai ... fasta file chr10.fa has 1 chromosomes/contigs

found match for reference contig chr10 in VCF file index 
contig chr10 length 133797422
reading reference sequence file chr10.fa with 1 contigs
read reference sequence file in 0.35 sec
reading sorted bam/cram file illumina.sorted.bam 
processing reads mapped to chrom "chr10" 


[2025:11:01 18:51:35] input fragment file: illumina.frag
[2025:11:01 18:51:35] input variantfile (VCF format):illumina.vcf
[2025:11:01 18:51:35] haplotypes will be output to file: illumina.hapcut
[2025:11:01 18:51:35] solution convergence cutoff: 5
[2025:11:01 18:51:35] read 275 variants from illumina.vcf file 
[2025:11:01 18:51:35] read fragment file and variant file: fragments 289 variants 275
mean num

Number of non-trivial connected components 7 max-Degree 235 connected variants 175 coverage-per-variant 44.914286 


## PART 5: Comparing VCF files

In [65]:
%%bash
bgzip illumina.hapcut.phased.VCF & bcftools index illumina.hapcut.phased.VCF.gz
bgzip pacbio.hapcut.phased.VCF & bcftools index pacbio.hapcut.phased.VCF.gz
bcftools isec -p variants illumina.hapcut.phased.VCF.gz pacbio.hapcut.phased.VCF.gz

# Number of variants that are shared
echo "$(grep -v '^#' variants/0002.vcf | wc -l) variants are shared."

# Number of variants that are unique to illumina
echo "$(grep -v '^#' variants/0000.vcf | wc -l) variants are unique to illumina."

# Number of variants that are unique to pacbio
echo "$(grep -v '^#' variants/0001.vcf | wc -l) variants are unique to pacbio."

     256 variants are shared.
      19 variants are unique to illumina.
      73 variants are unique to pacbio.


Finding variants that are not common for each gene of interest.

In [66]:
%%bash

bgzip variants/0000.vcf & bcftools index -f variants/0000.vcf.gz
bgzip variants/0001.vcf & bcftools index -f variants/0001.vcf.gz

# variants for CYP2C8 gene; region: chr10:95036772-95069497
bcftools view -r chr10:95036772-95069497 variants/0000.vcf.gz > variants/CYP2C8_variants_illumina.vcf
bcftools view -r chr10:95036772-95069497 variants/0001.vcf.gz > variants/CYP2C8_variants_pacbio.vcf

# variants for CYP2C9 gene; region: chr10:94938658-94990091
bcftools view -r chr10:94938658-94990091 variants/0000.vcf.gz > variants/CYP2C9_variants_illumina.vcf
bcftools view -r chr10:94938658-94990091 variants/0001.vcf.gz > variants/CYP2C9_variants_pacbio.vcf

# variants for CYP2C19 gene; region: chr10:94762681-94855547
bcftools view -r chr10:94762681-94855547 variants/0000.vcf.gz > variants/CYP2C19_variants_illumina.vcf
bcftools view -r chr10:94762681-94855547 variants/0001.vcf.gz > variants/CYP2C19_variants_pacbio.vcf

# Variants for CYP2C8 gene
## Variant found in position chrom10:95046748 from Illumina
![image](./images/95046748-illumina.png)

### Analysis
Illumina sequencing has a much lower read support for this particular position in the chromosome. Furthermore, some of the reads have different start positions around this position, indicating some instability. Also, in some of the strands, the deletions seem to 'slide' one or two positions in each direction which gives further support that this is likely a sequencing artifact.

## Variant found in position chrom10:95041650 from Pacbio
![image](./images/95041650-pacbio.png)

### Analysis
The variant specific to Pacbio sequencing has good reads support: 97 reads, 48 of which have an A nucleotide at this position, and the remaining have a G at this position. Since the entropy is low and the reads have good mapping quality (60 for most), and Illumina does not have enough reads (only 17) to contradict this variant call, I believe this is likely a true biological variant.

# Variants found in CYP2C9 gene
## Variant found in position chrom10:94947469 from Pacbio
![image](./images/94947469-pacbio.png)

### Analysis
Pacbio sequence has a much lower read depth at this position of the chromosome compared to Illumina. All of the reads in Illumina agree with the reference genome at this position. Also, 91% of the reads in the Pacbio sequence report a T nucleotide while only 3 reads have the alternative C nucleotide in this position, so this makes me believe it is a sequencing artifact and not a real biological variant.

## Variant found in position chrom10:94952992 from Pacbio
![image](./images/94952992-pacbio.png)

### Analysis
Similar to the previous variant, Pacbio sequencing has a lower read support compared to Illumina for this position in the genome. All the reads in Illumina report a G base at this position, agreeing with the reference genome. In the Pacbio sequence, only 3% of the reads have the alternative base call (A) which makes it likely that this was an sequencing error specific to the technology and not a true biological variant.

# Variants found in CYP2C19 gene
## Variant found in position chrom10:94772788 from Illumina
![image](./images/94772788-illumina.png)

### Analysis
All reads from the Illumina sequence at this position report the alternative base call (T) as opposed to the base call of the reference genome (g). Furthermore, there are no reads from Pacbio that align at this position, so there is no information that contradicts this variant calling. However, there are reads that start or end around this position, which may introduce some noise, this, along with the lack of evidence one way or the other from Pacbio, makes it hard to determine whether it is a sequencing error or a biological variant.

## Variants found in position chrom10:94770084 from Pacbio
![image](./images/94770084-pacbio.png)

### Analysis
The read depth at this position for the Pacbio alignment is really low (only 5). Furthermore, out of this 5 reads, only one has the alternative base call. Additionally, none of the reads from Illumina aligning at this position report this alternative base call, so it is likely that this is a sequencing error specific to Pacbio.